# The four sights, from the Buddha's own pixels

Prince Siddhartha left the palace and saw, for the first time, **an old man, a sick man, a dead man and an ascetic**. Those four sights set him on the path to becoming the Buddha.

Here each sight is painted with **exactly the pixels of a thangka of the Buddha**: every pixel used once, none added, none changed, only moved. Four paintings, one set of pixels. It's impermanence in arithmetic: the same stuff, rearranged into old age, sickness, death and renunciation.

**How.** The method from [`../same-pixels/same_pixels.ipynb`](../same-pixels/same_pixels.ipynb): Stable Diffusion 3.5 Medium samples with a custom loop. Partway through, the model's guess of the final picture is pulled onto the thangka's colors with sliced optimal transport and put back on the same noise path, so the model paints around the palette it's stuck with. At the end, an exact snap gives every output pixel a distinct thangka pixel. Since the palette is the thangka's, and the prompt asks for a thangka, the four come out in its style: flat mineral colors, fine outlines, turquoise sky and green hills, orange robes.

**Before running**

- *Runtime → Change runtime type → GPU.* A T4 works at 512 × 512; an L4 or A100 is faster and can go bigger.
- Accept the license at [huggingface.co/stabilityai/stable-diffusion-3.5-medium](https://huggingface.co/stabilityai/stable-diffusion-3.5-medium), and add a Hugging Face read token as a Colab secret named `HF_TOKEN` (the key icon on the left; turn on notebook access).
- Nothing to upload: the thangka is downloaded from where it was found.

In [ ]:
!pip install -q -U diffusers transformers accelerate sentencepiece protobuf

## Load the model

In [ ]:
import torch, numpy as np
from diffusers import StableDiffusion3Pipeline

device = "cuda"
# bf16 needs an Ampere-or-newer GPU (L4, A100). A T4 falls back to fp16.
dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16

# Skipping the T5 text encoder saves ~9 GB of GPU memory. Prompts still work through SD3's two CLIP
# encoders, with a bit less prompt adherence, so the prompts below are short and put the subject first.
pipe = StableDiffusion3Pipeline.from_pretrained(
    "stabilityai/stable-diffusion-3.5-medium",
    text_encoder_3=None, tokenizer_3=None, torch_dtype=dtype,
).to(device)
vae = pipe.vae

## Settings

- **W, H**: the output size, square. Both must be multiples of 16. On an L4 or A100, try 1024 × 1024.
- **STYLE**: added after every sight, so the four match each other and the thangka.
- **SIGHTS**: one prompt per sight. Subject first: CLIP reads only the first 77 tokens.
- **SEEDS**: one painting per sight per seed, to choose from.

In [ ]:
W, H = 512, 512
STYLE = ("Tibetan thangka painting, traditional Himalayan Buddhist art, flat mineral pigments, fine ink outlines, "
         "stylized clouds, green hills and turquoise sky, orange and red robes, decorative, symmetrical")
NEGATIVE = "photograph, photorealistic, realistic, 3d render, blurry, text, watermark, signature, frame"
SIGHTS = {
    "old_man":  "an old man with white hair and a long white beard, bent over a walking stick, wrinkled face",
    "sick_man": "a sick man lying on a mat, pale and feverish, eyes half closed, a companion tending him",
    "dead_man": "a dead man wrapped in a white shroud, carried on a bier by mourners in a funeral procession",
    "ascetic":  "a serene wandering ascetic with shaved head in saffron robes, holding an alms bowl, walking calmly",
}
STEPS = 28
CFG = 4.5
SEEDS = (0, 1)

## The thangka

The pixels every sight must use. The outputs are square but the thangka is 2:3, so `FIT` picks how it becomes W × H pixels:

- `"squeeze"` (default): the whole thangka, resized to the square. It looks squashed, but that doesn't matter: the sights only use its colors, rearranged, and squeezing keeps every part of the painting in the mix (the tree canopy and sky at the top, the offerings at the bottom).
- `"crop"`: the middle square only, undistorted. Loses most of the trees, sky and offerings, so the palette shifts toward the Buddha and his throne.

The sights are rearrangements of this resized version.

In [ ]:
import io, urllib.request
from PIL import Image, ImageOps

SOURCE_URL = ("https://cdn.sanity.io/images/caw6lfob/production/"
              "2bea4574756b533d9c2fdf209c21af5c6c923d49-2202x3259.jpg")
with urllib.request.urlopen(SOURCE_URL) as r:
    thangka = Image.open(io.BytesIO(r.read())).convert("RGB")
FIT = "squeeze"   # "squeeze" or "crop"
if FIT == "squeeze":
    source = thangka.resize((W, H), Image.LANCZOS)
else:
    source = ImageOps.fit(thangka, (W, H), Image.LANCZOS)
src8 = np.asarray(source)                               # (H, W, 3) uint8
src = torch.tensor(src8, device=device, dtype=torch.float32).reshape(-1, 3) / 255
print(thangka.size, "->", source.size)
source

## Helpers

From the same-pixels notebook, for any W × H.

- `decode` / `encode` convert between SD3's latent space and pixels. The histogram lives in pixel space, so every projection is a round trip.
- `sliced_ot` is the projection: rotate RGB space randomly, match each axis exactly by sorting, rotate back, repeat.
- `exact_snap` makes the output literally a permutation of the source's pixels. Pixels whose color is still available take it exactly; the rest are paired along a 3D Hilbert curve through color space.

In [ ]:
def decode(lat):
    lat = lat / vae.config.scaling_factor + vae.config.shift_factor
    return ((vae.decode(lat.to(vae.dtype)).sample.float() + 1) / 2).clamp(0, 1)

def encode(img):
    lat = vae.encode((img * 2 - 1).to(vae.dtype)).latent_dist.mean
    return (lat - vae.config.shift_factor) * vae.config.scaling_factor

def to_pixels(img):    # (1, 3, H, W) -> (H*W, 3)
    return img[0].permute(1, 2, 0).reshape(-1, 3)

def from_pixels(px):   # (H*W, 3) -> (1, 3, H, W)
    return px.reshape(H, W, 3).permute(2, 0, 1)[None]

def to_uint8(img):     # (1, 3, H, W) in [0, 1] -> (H, W, 3) uint8
    return (img[0].permute(1, 2, 0).cpu().numpy() * 255).round().astype(np.uint8)

def sliced_ot(x, src, iters=30):
    # Move x's color distribution onto src's. x, src: (n, 3) float tensors.
    x = x.clone()
    for _ in range(iters):
        Q, _ = torch.linalg.qr(torch.randn(3, 3, device=x.device))
        xp, sp = x @ Q, src @ Q
        for k in range(3):
            order = xp[:, k].argsort()
            xp[order, k] = sp[:, k].sort().values
        x = xp @ Q.T
    return x

def color_keys(c):
    c = c.astype(np.int64)
    return (c[:, 0] << 16) | (c[:, 1] << 8) | c[:, 2]

def hilbert(c, bits=8):
    # Position of each RGB color along a 3D Hilbert curve (Skilling's algorithm, vectorized).
    X = [c[:, i].astype(np.int64).copy() for i in range(3)]
    M = 1 << (bits - 1); Q = M
    while Q > 1:
        P = Q - 1
        for i in range(3):
            m = (X[i] & Q) != 0
            X[0] = np.where(m, X[0] ^ P, X[0])
            t = np.where(m, 0, (X[0] ^ X[i]) & P)
            X[0] ^= t; X[i] ^= t
        Q >>= 1
    for i in range(1, 3): X[i] ^= X[i - 1]
    t = np.zeros_like(X[0]); Q = M
    while Q > 1:
        t = np.where((X[2] & Q) != 0, t ^ (Q - 1), t); Q >>= 1
    for i in range(3): X[i] ^= t
    key = np.zeros(len(c), np.int64)
    for b in range(bits - 1, -1, -1):
        for i in range(3):
            key = (key << 1) | ((X[i] >> b) & 1)
    return key

def occurrence_ids(k):
    # Unique id per pixel: (color, how many times that color appeared before it).
    n = len(k); order = np.argsort(k, kind="stable"); ks = k[order]
    starts = np.r_[0, np.flatnonzero(np.diff(ks)) + 1]
    counts = np.diff(np.r_[starts, n])
    rank = np.empty(n, np.int64); rank[order] = np.arange(n) - np.repeat(starts, counts)
    return k * n + rank

def exact_snap(img, src):
    # Rearrange src's pixels to look as much like img as possible. Both (H, W, 3) uint8.
    x, s = img.reshape(-1, 3), src.reshape(-1, 3); n = len(s)
    ix, is_ = occurrence_ids(color_keys(x)), occurrence_ids(color_keys(s))
    s_order = np.argsort(is_); s_sorted = is_[s_order]
    pos = np.minimum(np.searchsorted(s_sorted, ix), n - 1)
    hit = s_sorted[pos] == ix
    perm = np.full(n, -1); perm[hit] = s_order[pos[hit]]
    used = np.zeros(n, bool); used[perm[hit]] = True
    lx, ls = np.flatnonzero(~hit), np.flatnonzero(~used)
    perm[lx[np.argsort(hilbert(x[lx]))]] = ls[np.argsort(hilbert(s[ls]))]
    return s[perm].reshape(src.shape), hit.mean()

def same_histogram(a, b):
    return np.array_equal(np.sort(color_keys(a.reshape(-1, 3))), np.sort(color_keys(b.reshape(-1, 3))))

## The sampling loop

As in the same-pixels notebook, with a negative prompt and any W × H. `diffusers` runs from noise (σ = 1) to image (σ = 0) with `x_t = (1 − σ)·image + σ·noise`, and the model predicts `v = noise − image`, so each step gives both the predicted image `x_t − σ·v` and the starting noise `x_t + (1 − σ)·v`. Inside `window`, every `project_every` steps, the predicted image is moved onto the thangka's colors and put back on the same noise path.

In [ ]:
@torch.no_grad()
def generate(prompt, project=True, project_every=3, window=(0.15, 0.9), seed=0):
    pe, npe, ppe, nppe = pipe.encode_prompt(prompt, prompt, prompt, device=device, negative_prompt=NEGATIVE,
                                            negative_prompt_2=NEGATIVE, negative_prompt_3=NEGATIVE)
    cond, pooled = torch.cat([npe, pe]), torch.cat([nppe, ppe])
    pipe.scheduler.set_timesteps(STEPS, device=device)
    sig = pipe.scheduler.sigmas
    c, f = pipe.transformer.config.in_channels, pipe.vae_scale_factor
    lat = torch.randn(1, c, H // f, W // f, device=device, dtype=dtype,
                      generator=torch.Generator(device).manual_seed(seed))
    for i, t in enumerate(pipe.scheduler.timesteps):
        out = pipe.transformer(hidden_states=torch.cat([lat] * 2), timestep=t.expand(2),
                               encoder_hidden_states=cond, pooled_projections=pooled, return_dict=False)[0]
        vu, vc = out.chunk(2)
        v = vu + CFG * (vc - vu)
        s = sig[i]
        if project and i % project_every == 0 and window[0] < s < window[1]:
            x0 = lat - s * v
            noise = lat + (1 - s) * v
            px = sliced_ot(to_pixels(decode(x0)), src).clamp(0, 1)
            x0 = encode(from_pixels(px)).to(dtype)
            lat = ((1 - s) * x0 + s * noise).to(dtype)
            v = noise - x0
        lat = (lat + (sig[i + 1] - s) * v).to(dtype)
    return decode(lat)

## Paint the four sights

One painting per sight per seed, each snapped to the thangka's exact pixels. Every result is checked: **identical** means every exact RGB color appears exactly as many times as in the thangka.

In [ ]:
import os
os.makedirs("sights", exist_ok=True)
Image.fromarray(src8).save("sights/thangka_source.png")
results = {}
for name, subject in SIGHTS.items():
    for seed in SEEDS:
        drawn = to_uint8(generate(f"{subject}, {STYLE}", seed=seed))
        snapped, exact = exact_snap(drawn, src8)
        change = np.abs(snapped.astype(int) - drawn.astype(int)).mean()
        results[name, seed] = snapped
        Image.fromarray(snapped).save(f"sights/{name}_seed{seed}.png")
        print(f"{name:>9} seed {seed}: snap moved pixels by {change:4.1f}/255 on average, {exact:.0%} kept exactly, "
              f"identical histogram: {same_histogram(snapped, src8)}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(SEEDS), len(SIGHTS) + 1, figsize=(4 * (len(SIGHTS) + 1), 4.4 * len(SEEDS)), squeeze=False)
for r, seed in enumerate(SEEDS):
    axes[r, 0].imshow(src8); axes[r, 0].set_title("the thangka")
    for c, name in enumerate(SIGHTS):
        axes[r, c + 1].imshow(results[name, seed]); axes[r, c + 1].set_title(f"{name.replace('_', ' ')}, seed {seed}")
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout(); plt.show()

## Pick and download

Choose a seed for each sight (or rerun a sight with new seeds: `results[name, seed]` for any new seed after rerunning the paint cell with `SEEDS` changed). The cell saves the four picks side by side with the thangka and downloads `four_sights.zip`: every painting, the source, and the picks.

In [ ]:
PICK = {"old_man": 0, "sick_man": 0, "dead_man": 0, "ascetic": 0}

row = Image.new("RGB", (5 * W + 4 * 16, H), "white")
for k, img in enumerate([src8] + [results[n, PICK[n]] for n in SIGHTS]):
    row.paste(Image.fromarray(img), (k * (W + 16), 0))
row.save("sights/four_sights.png")
for n in SIGHTS:
    Image.fromarray(results[n, PICK[n]]).save(f"sights/pick_{n}.png")
    assert same_histogram(results[n, PICK[n]], src8)

import shutil
shutil.make_archive("four_sights", "zip", "sights")
try:
    from google.colab import files
    files.download("four_sights.zip")
except ImportError:
    pass
row

## Things to try

- **More seeds** for the sight that's hardest (the dead man, probably: a white shroud has few white pixels to draw on).
- **`window` and `project_every`** in `generate`: projecting earlier makes the colors fit better but the drawing looser.
- **Bigger:** 1024 × 1024 on an L4 or A100.
- **If outputs are black or full of noise** on a T4, fp16 has probably overflowed. Switch to an L4 or A100 so the notebook uses bf16.